<a href="https://colab.research.google.com/github/kimjiwoo2/Pill-agent/blob/develop/notebooks/jiwoo/05_jw_cls_eda_manifest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **0. Overview**

### **탐색 및 초기 파이프라인**

- manifest 구조 확인, DB 조인, 속성 분포 EDA, 초기 학습 파이프라인 구성
- `pilliot_15k_v1_classification_manifest.csv` :single+combination, DB 조인 완료, 레이블 전처리 전 raw 상태
- 원본 이미지 드라이브 저장




# **1. Import**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import cv2
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
import torch

!pip install pymysql
import pymysql

# **2. Manifest Load**

In [ ]:
# Image ZIP
zip_path = '/content/drive/MyDrive/ToBigs/2425/Pillot/dataset/pilliot_15k_v1_final.zip'

# 내부 구조만 먼저 확인
with zipfile.ZipFile(zip_path, 'r') as z:
    names = z.namelist()
    print(f"총 파일 수: {len(names)}")
    print("\n상위 20개 경로:")
    for n in names[:20]:
        print(n)

In [ ]:
# Manifest CSV ZIP
manifest_path = '/content/drive/MyDrive/ToBigs/2425/Pillot/dataset/pilliot_manifests.zip'

with zipfile.ZipFile(manifest_path, 'r') as z:
    print("내부 파일 목록:")
    for name in z.namelist():
        print(name)

In [ ]:
with zipfile.ZipFile(manifest_path, 'r') as z:
    with z.open('classification_single_manifest_all.csv') as f:
        df = pd.read_csv(f, nrows=5)
        print(df.columns.tolist())
        print(df.head())

In [ ]:
with zipfile.ZipFile(manifest_path, 'r') as z:
    with z.open('detection_single.csv') as f:
        df = pd.read_csv(f, nrows=5)
        print(df.columns.tolist())
        print(df.head())

In [ ]:
# Manifest에서 이미지 파일 필터링
manifest_path = '/content/drive/MyDrive/ToBigs/2425/Pillot/dataset/pilliot_manifests.zip'
dataset_path = '/content/drive/MyDrive/ToBigs/2425/Pillot/dataset/pilliot_15k_v1_final.zip'

zip12_single = [
    'TL_81_단일.zip', 'TL_76_단일.zip', 'TL_46_단일.zip', 'TL_38_단일.zip',
    'TL_48_단일.zip', 'TL_43_단일.zip', 'TL_70_단일.zip', 'TL_73_단일.zip',
    'TL_54_단일.zip', 'TL_41_단일.zip', 'TL_75_단일.zip', 'TL_57_단일.zip'
]

# 실제 이미지 파일명 → zip 내부 전체 경로 매핑
with zipfile.ZipFile(dataset_path, 'r') as z:
    path_map = {}
    actual_files = set()
    for name in z.namelist():
        if name.endswith('.png') or name.endswith('.jpg'):
            filename = name.split('/')[-1]
            path_map[filename] = name
            actual_files.add(filename)

print(f"실제 이미지 파일 수: {len(actual_files)}")

# single manifest 로드 및 필터링
with zipfile.ZipFile(manifest_path, 'r') as z:
    with z.open('detection_single.csv') as f:
        df_single = pd.read_csv(f)

df_single = df_single[df_single['label_zip_name'].isin(zip12_single)]
df_single = df_single[df_single['image_file'].isin(actual_files)]
print(f"single 매칭: {len(df_single)}행")

# combination manifest 로드 및 필터링
with zipfile.ZipFile(manifest_path, 'r') as z:
    with z.open('detection_combination.csv') as f:
        df_comb = pd.read_csv(f)

df_comb = df_comb[df_comb['image_file'].isin(actual_files)]
print(f"combination 매칭: {len(df_comb)}행")

# 합치기
df_cls_final = pd.concat([df_single, df_comb], ignore_index=True)

# zip 내부 전체 경로 컬럼 추가
df_cls_final['zip_path'] = df_cls_final['image_file'].map(path_map)

print(f"최종: {len(df_cls_final)}행")
print(f"매핑 실패: {df_cls_final['zip_path'].isna().sum()}행")
print(df_cls_final[['image_file', 'zip_path', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h']].head(3))

# **3. DB 조인**

In [ ]:
# Manifest CSV에는 속성 분류를 위한 컬럼 존재하지 않음
print(df_cls_final.columns.tolist())

In [ ]:
conn = pymysql.connect(
    host='103.218.161.72',
    port=3306,
    user='jiwoo_admin',
    password='1234',
    database='pilliot_db',
    charset='utf8mb4'
)

item_seqs_str = ','.join(map(str, [int(x) for x in df_cls_final['item_seq'].dropna().unique()]))

query = f"""
SELECT item_seq, dl_mapping_code,
       drug_shape, color_class1, color_class2,
       form_code_name, line_front, line_back,
       print_front, print_back,
       chart, di_class_no, di_etc_otc_code, di_edi_code
FROM drug_master
WHERE item_seq IN ({item_seqs_str})
"""
df_drug = pd.read_sql(query, conn)

df_cls_final = df_cls_final.merge(df_drug, on='item_seq', how='left')
print(f"조인 결과: {len(df_cls_final)}행")
print(df_cls_final.columns.tolist())

# 저장: pilliot_15k_v1_classification_manifest.csv
# single + combination 전체 (23,494행), DB 조인 완료
# 레이블 전처리 전 raw 상태
df_cls_final.to_csv('/content/drive/MyDrive/ToBigs/2425/Pillot/dataset/pilliot_15k_v1_full_manifest_raw.csv', index=False)
print("저장 완료")

- **`item_seq` — 품목 고유 식별자 (Primary Key). 식약처 품목 코드.**

- **`dl_mapping_code` — 식약처 의약품 매핑 코드. 외부 DB 연동 시 사용.**

- **`dl_name` — 약품 한글 이름. 예: 타이레놀정500mg.**

- `dl_name_en` — 약품 영문 이름.

- `dl_company` — 제조사/판매사 이름. 예: 한국얀센.

- `dl_material` — 주성분 한글명. 예: 아세트아미노펜.

- `dl_material_en` — 주성분 영문명. 예: Acetaminophen.

- **`drug_shape` — 알약 모양. 원형/타원형/장방형 등.**

- **`color_class1` — 주색상.**

- **`color_class2` — 부색상. 투톤일 때만 값 있음.**

- **`form_code_name` — 제형. 정제/캡슐/연질캡슐 등.**

- `print_front` — 앞면 각인 텍스트. OCR 정답 레이블로 활용 가능.

- `print_back` — 뒷면 각인 텍스트.  OCR 정답 레이블로 활용 가능.

- `line_front` — 앞면 분할선 유무/형태.

- `line_back` — 뒷면 분할선 유무/형태.

- `chart` — 약품 외형 설명 텍스트. 색상/모양/크기 등 자연어로 기술된 정보.

- `di_class_no` — 약효 분류 번호. 어떤 질환에 쓰이는 약인지 분류.

- `di_etc_otc_code` — 전문의약품/일반의약품 구분 코드.

- `di_edi_code` — 건강보험 EDI 코드. DUR 데이터 연동 시 핵심 키값.

# **4. EDA**
- single vs combination 분포 비교
- 각 속성 (drug_shape, color_class1, form_code_name, line_front) 분포 확인
- bbox 크기 분포
- 품목 수 확인

In [ ]:
# combination vs single 분포 비교
df_single_only = df_cls_final[df_cls_final['dataset_type'] == 'single']
df_comb_only = df_cls_final[df_cls_final['dataset_type'] == 'combination']

print(f"single: {len(df_single_only)}행 / combination: {len(df_comb_only)}행")

for col in ['drug_shape', 'color_class1', 'form_code_name', 'line_front']:
    print(f"\n[{col}]")
    print(pd.DataFrame({
        'single': df_single_only[col].value_counts(normalize=True).round(3),
        'combination': df_comb_only[col].value_counts(normalize=True).round(3)
    }).fillna(0))

In [ ]:
# 각 속성 고유값 및 분포 확인
for col in ['drug_shape', 'color_class1', 'form_code_name', 'line_front']:
    print(f"\n[{col}] 고유값 수: {df_cls_final[col].nunique()}")
    print(df_cls_final[col].value_counts(dropna=False))

-> combination으로부터 파생된 이미지로 인해 single 기준 EDA 분포가 깨짐.

-> 우선 첫 번째 단계에서는 single 이미지만을 활용하여 분류기 학습하기로 결정

In [ ]:
single_items = set(df_cls_final[df_cls_final['dataset_type'] == 'single']['item_seq'].unique())
comb_items = set(df_cls_final[df_cls_final['dataset_type'] == 'combination']['item_seq'].unique())

print(f"manifest 전체 행 수: {len(df_cls_final)}")
print(f"고유 item_seq 수: {df_cls_final['item_seq'].nunique()}")
print(f"single 품목 수: {len(single_items)}")
print(f"combination 품목 수: {len(comb_items)}")
print(f"combination에만 있는 품목 수: {len(comb_items - single_items)}")

In [ ]:
# bbox 크기 분포
df_single['bbox_area'] = df_single['bbox_w'] * df_single['bbox_h']

print(df_single[['bbox_w', 'bbox_h', 'bbox_area']].describe())

# 너무 작은 bbox 확인
small_bbox = df_single[df_single['bbox_area'] < 50*50]
print(f"\nbbox 50x50 미만: {len(small_bbox)}행")

In [ ]:
# color_class1 속성 - 복합값 알약 샘플 확인
complex_colors = df_single[df_single['color_class1'].str.contains(',', na=False)]
print(f"복합값 총 행 수: {len(complex_colors)}")
print(f"고유 품목 수: {complex_colors['item_seq'].nunique()}")
print("\n샘플:")
print(complex_colors[['item_seq', 'dl_name', 'color_class1', 'drug_shape', 'form_code_name']].drop_duplicates('item_seq').head(10))

In [ ]:
# 투명 계열 알약 이미지 샘플 시각화
complex_colors = df_single[df_single['color_class1'].str.contains(',', na=False)]
samples = complex_colors.sample(9, random_state=42)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
with zipfile.ZipFile(dataset_path, 'r') as z:
    for ax, (_, row) in zip(axes.flatten(), samples.iterrows()):
        with z.open(row['zip_path']) as f:
            img = Image.open(f)
            img.load()
        x, y, w, h = int(row['bbox_x']), int(row['bbox_y']), int(row['bbox_w']), int(row['bbox_h'])
        pad_x, pad_y = int(w * 0.1), int(h * 0.1)
        img_w, img_h = img.size
        x1 = max(0, x - pad_x)
        y1 = max(0, y - pad_y)
        x2 = min(img_w, x + w + pad_x)
        y2 = min(img_h, y + h + pad_y)
        cropped = img.crop((x1, y1, x2, y2))
        ax.imshow(cropped)
        ax.set_title(f"{row['color_class1']}\n{row['dl_name'][:15]}", fontsize=8)
        ax.axis('off')

plt.tight_layout()
plt.show()

# **5. 원본 이미지 압축 해제 및 저장**

In [ ]:
# 압축 해제
extract_path = '/content/drive/MyDrive/ToBigs/2425/Pillot/dataset/pilliot_15k_v1_final/'

if not os.path.exists(extract_path):
    print("압축 해제 중...")
    with zipfile.ZipFile(dataset_path, 'r') as z:
        z.extractall(extract_path)
    print("완료")
else:
    print("이미 압축 해제됨")